In [1]:
!pip install -U transformers accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 102.5 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0

In [2]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [3]:
! pip install --upgrade datasets pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 41.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.3
    Uninstalling datasets-4.8.3:
      Successfully uninstalled datasets-4.8.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [4]:
! pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 76.3 MB/s eta 0:00:00


In [5]:
import os, re, gc, time, shutil, json
import numpy as np
import pandas as pd
import torch
import librosa
import evaluate

from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from pathlib import Path

from datasets import (
    Dataset, DatasetDict,
    Array2D, Sequence, Value, Features,
    Audio as HFAudio,
)
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

# ── GPU diagnostics ──────────────────────────────────────────────────────────
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM     : {vram:.1f} GB')
    print(f'sm_cap   : sm_{cap[0]}{cap[1]}')
    # Quick tensor test — fails immediately if CUDA ABI mismatch
    try:
        _ = torch.randn(4, 4).cuda() @ torch.randn(4, 4).cuda()
        print('GPU test : ✅ OK')
    except Exception as e:
        print(f'GPU test : ❌ FAILED — {e}')
        print('Switch accelerator to T4 in Kaggle settings!')

PyTorch  : 2.10.0+cu128
CUDA     : 12.8
GPU      : Tesla T4
VRAM     : 15.6 GB
sm_cap   : sm_75
GPU test : ✅ OK


In [6]:
CFG = {
    # ── Dataset ──────────────────────────────────────────────────────
    # Root folder of your uploaded Common Voice 25.0 Arabic dataset
    'data_root': '/kaggle/input/datasets/omar10lfc/common-voice-scripted-speech-25-0-arabic',

    # How many TRAINING samples to use.
    # Common Voice Arabic train.tsv has 28,865 rows.
    # Recommended sizes:
    #   Quick test  : 1_000  (~15 min feature extraction, ~20 min training)
    #   Good result : 5_000  (~45 min feature extraction, ~1 hr training)
    #   Full run    : 28_865 (~3 hrs feature extraction,  ~4 hrs training)
    'n_train': 25000,

    # How many TEST samples to evaluate on.
    # Keeping this small makes eval fast (WER computation is slow).
    'n_test': 300,

    # Random seed — controls shuffle order (reproducible)
    'seed': 42,

    # ── Model ─────────────────────────────────────────────────────────
    # 'openai/whisper-tiny'   — 39M  params, fastest,  lowest quality
    # 'openai/whisper-small'  — 244M params, balanced  ← recommended
    # 'openai/whisper-medium' — 769M params, best quality, needs 16GB+
    'model_name': 'openai/whisper-small',

    # ── Training ──────────────────────────────────────────────────────
    # Tuned for ~20s WER in less wall time:
    #   - LR 1e-5 is the standard for whisper-small fine-tuning (HF blog).
    #     5e-6 was too cold; convergence was slow.
    #   - 1200 steps × eff-batch 32 ≈ 3.8 passes over 10k clips —
    #     enough at the higher LR. More was marginal gains for big compute.
    #   - eval/save every 400 → 3 checkpoints, so early-stopping + best-model
    #     selection actually find the right point instead of overshooting.
    'max_steps': 2000,
    'eval_steps': 500,
    'save_steps': 500,
    'logging_steps': 25,
    'batch_size': 8,           # Per-device batch size
    'grad_accumulation': 4,     # Effective batch = batch_size * grad_accumulation
    'learning_rate': 1e-5,
    'warmup_steps': 500,
    'weight_decay': 0.01,
    'num_workers': 2,
    'early_stopping_patience': 3,  # Stop if WER doesn't improve for N evals

    # ── Paths ─────────────────────────────────────────────────────────
    'output_dir':   '/kaggle/working/whisper-arabic',
    'cache_dir':    '/kaggle/tmp/whisper_cache',
}

# Create output dirs
os.makedirs(CFG['output_dir'], exist_ok=True)
os.makedirs(CFG['cache_dir'],  exist_ok=True)

print('Configuration:')
for k, v in CFG.items():
    print(f'  {k:25s}: {v}')

Configuration:
  data_root                : /kaggle/input/datasets/omar10lfc/common-voice-scripted-speech-25-0-arabic
  n_train                  : 25000
  n_test                   : 300
  seed                     : 42
  model_name               : openai/whisper-small
  max_steps                : 2000
  eval_steps               : 500
  save_steps               : 500
  logging_steps            : 25
  batch_size               : 8
  grad_accumulation        : 4
  learning_rate            : 1e-05
  warmup_steps             : 500
  weight_decay             : 0.01
  num_workers              : 2
  early_stopping_patience  : 3
  output_dir               : /kaggle/working/whisper-arabic
  cache_dir                : /kaggle/tmp/whisper_cache


In [7]:
# WHY: Common Voice 25.0 is split across 1900+ numbered subdirectories.
# We do ONE os.walk() pass to build a lookup dict: filename_stem → full_path.
# This is much faster than searching the tree per sample.

print(f'Scanning {CFG["data_root"]} ...')
t0 = time.time()

tsv_index   = {}   # 'train' / 'dev' / 'test' → full .tsv path
audio_index = {}   # 'common_voice_ar_XXXXX'  → full .mp3 path

for root, _, files in os.walk(CFG['data_root']):
    for fname in files:
        full = os.path.join(root, fname)
        if fname.endswith('.tsv'):
            stem = fname[:-4]   # remove .tsv
            if stem not in tsv_index:
                tsv_index[stem] = full
        elif fname.endswith(('.mp3', '.wav', '.ogg', '.m4a')):
            stem = os.path.splitext(fname)[0]
            audio_index[stem] = full

print(f'Scan done in {time.time()-t0:.1f}s')
print(f'TSV files : {list(tsv_index.keys())}')
print(f'Audio clips: {len(audio_index):,}')

Scanning /kaggle/input/datasets/omar10lfc/common-voice-scripted-speech-25-0-arabic ...
Scan done in 311.9s
TSV files : ['test', 'dev', 'train', 'clip_durations', 'unvalidated_sentences', 'validated_sentences', 'invalidated', 'other', 'validated', 'reported']
Audio clips: 136,140


In [8]:
# WHY normalize_arabic: Whisper outputs normalized text but ground truth has
# diacritics, alef variants, etc. Computing WER on raw text inflates the score.
# We normalize BOTH hypothesis and reference before any WER computation.

def normalize_arabic(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[إأآٱ]', 'ا', text)              # alef variants → ا
    text = re.sub(r'ى', 'ي', text)                    # ya variants → ي
    text = re.sub(r'ة', 'ه', text)                    # ta marbuta → ه
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text) # strip diacritics
    text = re.sub(r'ـ', '', text)                      # strip tatweel
    text = re.sub(r'[^\w\s\u0600-\u06FF]', '', text)  # keep Arabic + alphanum
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def load_tsv(key: str, n_samples: Optional[int] = None, seed: int = 42) -> pd.DataFrame:
    """
    Load a TSV split, resolve audio paths, normalize text.
    Shuffles BEFORE subsampling so the subset is representative.
    """
    if key not in tsv_index:
        raise FileNotFoundError(f'{key}.tsv not found. Available: {list(tsv_index)}')

    df = pd.read_csv(tsv_index[key], sep='\t', low_memory=False)
    print(f'[{key}] Loaded {len(df):,} rows')

    # Resolve columns — CV 25.0 uses 'path' and 'sentence'
    path_col = 'path' if 'path' in df.columns else df.columns[1]
    text_col = 'sentence' if 'sentence' in df.columns else 'text'

    # Resolve audio paths
    def resolve(raw):
        stem = os.path.splitext(os.path.basename(str(raw)))[0]
        return audio_index.get(stem)

    df['audio_path'] = df[path_col].apply(resolve)
    df['sentence']   = df[text_col].apply(normalize_arabic)

    # Drop rows with missing audio or empty text
    before = len(df)
    df = df.dropna(subset=['audio_path'])
    df = df[df['sentence'].str.len() > 2]
    df = df[['audio_path', 'sentence']].reset_index(drop=True)
    print(f'  → {len(df):,} valid rows (dropped {before - len(df):,})')

    # Shuffle BEFORE subsampling — ensures representative subset
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    # Subsample
    if n_samples and n_samples < len(df):
        df = df.iloc[:n_samples].reset_index(drop=True)
        print(f'  → Subsampled to {len(df):,} rows')

    return df


# Load train (combine train + dev for more data), test separately
df_train_raw = load_tsv('train', seed=CFG['seed'])
df_dev_raw   = load_tsv('dev',   seed=CFG['seed'])
df_test_raw  = load_tsv('test',  n_samples=CFG['n_test'], seed=CFG['seed'])

# Combine train + dev, shuffle again, then subsample
df_all = pd.concat([df_train_raw, df_dev_raw], ignore_index=True)
df_all = df_all.sample(frac=1, random_state=CFG['seed']).reset_index(drop=True)
df_train = df_all.iloc[:CFG['n_train']].reset_index(drop=True)

print(f'\n✅ Final split sizes:')
print(f'  Train : {len(df_train):,}')
print(f'  Test  : {len(df_test_raw):,}')
print(f'\nSample:')
print(df_train.iloc[0])

[train] Loaded 28,865 rows
  → 28,862 valid rows (dropped 3)
[dev] Loaded 10,229 rows
  → 10,227 valid rows (dropped 2)
[test] Loaded 10,508 rows
  → 10,506 valid rows (dropped 2)
  → Subsampled to 300 rows

✅ Final split sizes:
  Train : 25,000
  Test  : 300

Sample:
audio_path    /kaggle/input/datasets/omar10lfc/common-voice-...
sentence                                           قتلت بتي امه
Name: 0, dtype: object


In [9]:
# WHY load model before feature extraction:
# The processor defines the feature extractor config (n_mels, hop_length etc.)
# We need it to extract features correctly.

print(f'Loading {CFG["model_name"]}...')

processor = WhisperProcessor.from_pretrained(
    CFG['model_name'],
    language='Arabic',
    task='transcribe',
)

model = WhisperForConditionalGeneration.from_pretrained(CFG['model_name'])

# Force Arabic — without this Whisper may auto-detect language
# and switch to English mid-batch, producing garbage outputs.
model.generation_config.language = 'arabic'
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None

print(f'Parameters: {model.num_parameters():,}')
print(f'Encoder layers: {model.config.encoder_layers}')
print(f'Decoder layers: {model.config.decoder_layers}')

Loading openai/whisper-small...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Parameters: 241,734,912
Encoder layers: 12
Decoder layers: 12


In [10]:
# WHY chunk + save to /kaggle/tmp:
# Processing all 5000 samples at once loads ~5GB of float32 audio into RAM,
# which crashes the kernel. Chunks of 300 keep peak RAM under 2GB.
# /kaggle/tmp has a separate quota from /kaggle/working.
#
# WHY explicit Array2D schema:
# Without it, HuggingFace stores spectrograms as nested Python lists,
# making DataLoader access ~50x slower than Arrow-backed arrays.

ARROW_FEATURES = Features({
    'input_features': Array2D(shape=(80, 3000), dtype='float32'),
    'labels':         Sequence(Value('int64')),
})


def extract_features(
    df: pd.DataFrame,
    split_name: str,
    chunk_size: int = 300,
) -> Dataset:
    """
    Extract log-mel features + token labels from a dataframe.
    Processes in chunks to keep RAM flat.
    Skips chunks already on disk (safe to re-run after crash).
    """
    cache = os.path.join(CFG['cache_dir'], split_name)
    os.makedirs(cache, exist_ok=True)

    total    = len(df)
    n_chunks = (total + chunk_size - 1) // chunk_size
    done_paths = []
    skipped = 0

    for i in range(n_chunks):
        chunk_path = os.path.join(cache, f'chunk_{i:04d}')

        # Resume: skip already-processed chunks
        if os.path.exists(os.path.join(chunk_path, 'dataset_info.json')):
            done_paths.append(chunk_path)
            skipped += 1
            continue

        start = i * chunk_size
        end   = min(start + chunk_size, total)
        rows  = df.iloc[start:end]

        feats_list  = []
        labels_list = []

        for _, row in rows.iterrows():
            try:
                # Load + resample to 16kHz — Whisper requirement
                audio, _ = librosa.load(row['audio_path'], sr=16000, mono=True)

                # Skip clips outside 0.5–29s range
                # (too short = mostly noise; >29s gets truncated by Whisper)
                duration = len(audio) / 16000
                if not (0.5 <= duration <= 29.0):
                    continue

                # Log-mel spectrogram: (80, 3000) fixed size
                feat = processor.feature_extractor(
                    audio,
                    sampling_rate=16000,
                ).input_features[0].astype(np.float32)

                # Tokenize transcript
                token_ids = processor.tokenizer(
                    row['sentence']
                ).input_ids

                feats_list.append(feat)
                labels_list.append(token_ids)

            except Exception:
                continue   # skip corrupted files silently

        if not feats_list:
            continue

        # Build Arrow dataset with explicit schema — critical for speed
        chunk_ds = Dataset.from_dict(
            {
                'input_features': np.array(feats_list, dtype=np.float32),
                'labels':         labels_list,
            },
            features=ARROW_FEATURES,
        )
        chunk_ds.save_to_disk(chunk_path)
        done_paths.append(chunk_path)

        # Explicitly free memory
        del feats_list, labels_list, chunk_ds
        gc.collect()

        pct = end / total * 100
        print(f'  [{split_name}] {i+1}/{n_chunks} — {end:,}/{total:,} ({pct:.0f}%)',
              flush=True)

    if skipped:
        print(f'  [{split_name}] Skipped {skipped} already-done chunks (resume)')

    # Load and concatenate all chunks
    print(f'  [{split_name}] Concatenating {len(done_paths)} chunks...')
    from datasets import concatenate_datasets
    all_chunks = [Dataset.load_from_disk(p) for p in done_paths]
    result = concatenate_datasets(all_chunks)
    
    # REMOVED: result = result.with_format('numpy') 
    # Bypassing this prevents the internal numpy=2.0 crash in Hugging Face datasets.

    # Clean up chunks — data is now in memory
    for p in done_paths:
        shutil.rmtree(p, ignore_errors=True)

    print(f'  [{split_name}] ✅ {len(result):,} samples ready')
    return result


print('=== Extracting TRAIN features ===')
train_dataset = extract_features(df_train,   'train')

print('\n=== Extracting TEST features ===')
test_dataset  = extract_features(df_test_raw, 'test')

dataset = DatasetDict({'train': train_dataset, 'test': test_dataset})

print(f'\n✅ Feature extraction complete')
print(f'  Train : {len(dataset["train"]):,}')
print(f'  Test  : {len(dataset["test"]):,}')
# Safely convert to numpy array using np.asarray() as recommended by NumPy 2.0+
print(f'  Shape : {np.asarray(dataset["train"][0]["input_features"]).shape}')

=== Extracting TRAIN features ===


Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

  [train] 1/84 — 300/25,000 (1%)


Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

  [train] 2/84 — 600/25,000 (2%)


...

  [train] 83/84 — 24,900/25,000 (100%)


Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

  [train] 84/84 — 25,000/25,000 (100%)
  [train] Concatenating 84 chunks...
  [train] ✅ 25,000 samples ready

=== Extracting TEST features ===


Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

  [test] 1/1 — 300/300 (100%)
  [test] Concatenating 1 chunks...
  [test] ✅ 300 samples ready

✅ Feature extraction complete
  Train : 25,000
  Test  : 300
  Shape : (80, 3000)


In [11]:
# WHY a custom collator:
# input_features are fixed-size (80×3000) — just stack them.
# labels are variable-length — pad to the longest in the batch.
# Padding positions are replaced with -100 so CrossEntropyLoss ignores them.

@dataclass
class DataCollatorWhisper:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Stack spectrograms — already same shape, no padding needed
        input_features = [
            {'input_features': f['input_features']} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors='pt'
        )

        # Pad label sequences to uniform length within this batch
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch   = self.processor.tokenizer.pad(
            label_features, return_tensors='pt'
        )

        # Replace pad token with -100 → ignored by loss
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Strip decoder start token if present (model adds it itself)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch


data_collator = DataCollatorWhisper(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)
print('Data collator ready')

Data collator ready


In [12]:
# WHY WER over loss for model selection:
# Validation loss measures next-token prediction probability,
# but WER measures actual transcription quality — what we care about.
# A model with lower loss can have HIGHER WER if it memorizes token patterns
# but fails to generalize to the actual word sequence.

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Restore pad tokens so the tokenizer can decode properly
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # Normalize BOTH sides — same function used on ground truth at training time
    # This prevents WER inflation from diacritic differences
    pred_str  = [normalize_arabic(s) for s in pred_str]
    label_str = [normalize_arabic(s) for s in label_str]

    # Filter empty predictions (can happen on very short/silent clips)
    pairs = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not pairs:
        return {'wer': 100.0}

    preds, labels = zip(*pairs)
    wer = 100 * wer_metric.compute(predictions=list(preds), references=list(labels))
    return {'wer': round(wer, 2)}


print('WER metric ready')

WER metric ready


In [13]:
# Detect if we can use bf16 (Ampere+ GPUs) vs fp16 (T4/V100)
# use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
# use_fp16 = torch.cuda.is_available() and not use_bf16
# print(f'Mixed precision: {"bf16" if use_bf16 else "fp16" if use_fp16 else "none (CPU)"}')
use_bf16 = False
use_fp16 = True
print(f'Mixed precision forced to: fp16 for T4 optimization')
# Check for existing checkpoint to resume from
resume_checkpoint = None
if os.path.isdir(CFG['output_dir']):
    checkpoints = [
        d for d in os.listdir(CFG['output_dir'])
        if d.startswith('checkpoint-')
    ]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1]
        resume_checkpoint = os.path.join(CFG['output_dir'], latest)
        print(f'Resuming from checkpoint: {resume_checkpoint}')
    else:
        print('No checkpoint found — training from scratch')


# Dropout zeroed: short fine-tuning runs on a clean scripted-speech corpus
# don't need extra regularization, and dropout slows convergence.
model.config.activation_dropout = 0.05
model.config.attention_dropout = 0.05
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05
training_args = Seq2SeqTrainingArguments(
    output_dir=CFG['output_dir'],

    # ── Batch ──────────────────────────────────────────────────────────
    per_device_train_batch_size=CFG['batch_size'],
    gradient_accumulation_steps=CFG['grad_accumulation'],
    # Bigger eval batch → faster generation passes (eval is the bottleneck once
    # gradient checkpointing is off).
    per_device_eval_batch_size=CFG['batch_size'],

    # ── Schedule ───────────────────────────────────────────────────────
    learning_rate=CFG['learning_rate'],
    warmup_steps=CFG['warmup_steps'],
    lr_scheduler_type='cosine',   # linear decay helps avoid overfitting on small sets
    max_steps=CFG['max_steps'],
    weight_decay=CFG['weight_decay'],

    # ── Eval & saving ──────────────────────────────────────────────────
    eval_strategy='steps',
    eval_steps=CFG['eval_steps'],
    save_strategy='steps',
    save_steps=CFG['save_steps'],
    save_total_limit=2,           # keep only last 2 checkpoints → saves disk
    logging_steps=CFG['logging_steps'],

    # ── Best model selection ───────────────────────────────────────────
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,      # lower WER is better

    # ── Generation ─────────────────────────────────────────────────────
    predict_with_generate=True,
    # CV Arabic transcripts are short (~10-15 tokens). 80 is plenty and roughly
    # halves eval generation time vs. 128.
    generation_max_length=80,

    # ── Memory / speed ─────────────────────────────────────────────────
    fp16=use_fp16,
    bf16=use_bf16,
    # Gradient checkpointing OFF: whisper-small fits comfortably on T4 (16GB)
    # at batch 16 fp16 without it. Turning it off is ~25-30% faster wall time.
    gradient_checkpointing=False,
    optim='adamw_torch',

    # ── Data loading ───────────────────────────────────────────────────
    # num_workers=0: dataset is in RAM, workers add overhead not speed
    dataloader_num_workers=CFG['num_workers'],
    dataloader_pin_memory=True,
    remove_unused_columns=False,

    # ── Misc ───────────────────────────────────────────────────────────
    report_to=['tensorboard'],
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[
        # Stop early if WER doesn't improve — prevents overfitting on small sets
        EarlyStoppingCallback(
            early_stopping_patience=CFG['early_stopping_patience']
        )
    ],
)

eff_batch = CFG['batch_size'] * CFG['grad_accumulation']
print(f'Trainer ready')
print(f'  Effective batch size : {eff_batch}')
print(f'  Max steps            : {CFG["max_steps"]}')
print(f'  Eval every           : {CFG["eval_steps"]} steps')
print(f'  Resume checkpoint    : {resume_checkpoint or "None"}')

Mixed precision forced to: fp16 for T4 optimization
No checkpoint found — training from scratch


2026-04-29 09:15:56.523108: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777454156.961635      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777454157.090395      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777454158.195642      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777454158.195675      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777454158.195680      23 computation_placer.cc:177] computation placer alr

Trainer ready
  Effective batch size : 32
  Max steps            : 2000
  Eval every           : 500 steps
  Resume checkpoint    : None


In [14]:
# WHY evaluate before training:
# This gives you the baseline WER of the pretrained Whisper-small on Arabic.
# The difference between baseline and post-training WER is your key CV metric.
print('Evaluating baseline WER (pretrained, no fine-tuning)...')
print('This takes ~3–5 minutes on T4...')

# 1. Find and temporarily remove the notebook callback (no import needed)
notebook_callback = None
for callback in trainer.callback_handler.callbacks:
    if callback.__class__.__name__ == 'NotebookProgressCallback':
        notebook_callback = callback
        break

if notebook_callback is not None:
    trainer.remove_callback(notebook_callback)

# 2. Run the evaluation
baseline = trainer.evaluate()
BASELINE_WER = baseline['eval_wer']

# 3. Add the exact callback instance back for training
if notebook_callback is not None:
    trainer.add_callback(notebook_callback)

print(f'\n{"="*45}')
print(f'  BASELINE WER : {BASELINE_WER:.2f}%')
print(f'{"="*45}')
print('Record this number — it goes on your CV.')

Evaluating baseline WER (pretrained, no fine-tuning)...
This takes ~3–5 minutes on T4...


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform


  BASELINE WER : 42.69%
Record this number — it goes on your CV.


In [15]:
# WHY resume_from_checkpoint:
# Kaggle sessions can time out. If the notebook restarts and checkpoints exist,
# training picks up from the last saved step rather than starting over.
# The `resume_checkpoint` variable was set in Cell 8 to the latest checkpoint.

print('Starting fine-tuning...')
eff_batch = CFG['batch_size'] * CFG['grad_accumulation']
samples_per_step = eff_batch
total_samples    = CFG['max_steps'] * samples_per_step
print(f'  Will process ~{total_samples:,} samples across {CFG["max_steps"]} steps')
print(f'  (dataset has {len(dataset["train"]):,} samples = '
      f'{total_samples / len(dataset["train"]):.1f} passes through data)')

train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

print('\nFine-tuning complete!')
print(f'  Train runtime : {train_result.metrics["train_runtime"]:.0f}s '
      f'({train_result.metrics["train_runtime"]/3600:.1f}h)')
print(f'  Train loss    : {train_result.metrics["train_loss"]:.4f}')

Starting fine-tuning...
  Will process ~64,000 samples across 2000 steps
  (dataset has 25,000 samples = 2.6 passes through data)


Step,Training Loss,Validation Loss,Wer
500,2.397821,0.325147,28.530000
1000,1.476885,0.262407,26.930000
1500,0.993666,0.258265,22.210000
2000,0.756624,0.251730,21.310000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].



Fine-tuning complete!
  Train runtime : 32096s (8.9h)
  Train loss    : 2.0911


In [16]:
print('Evaluating fine-tuned model...')
final = trainer.evaluate()
FINAL_WER = final['eval_wer']

print(f'\n{"="*45}')
print(f'  BASELINE WER   : {BASELINE_WER:.2f}%')
print(f'  FINE-TUNED WER : {FINAL_WER:.2f}%')
abs_improvement = BASELINE_WER - FINAL_WER
rel_improvement = abs_improvement / BASELINE_WER * 100
print(f'  IMPROVEMENT    : {abs_improvement:.2f}pp absolute  '
      f'({rel_improvement:.1f}% relative)')
print(f'{"="*45}')

# Save results to JSON for your README
results = {
    'model':          CFG['model_name'],
    'n_train':        CFG['n_train'],
    'n_test':         CFG['n_test'],
    'max_steps':      CFG['max_steps'],
    'baseline_wer':   round(BASELINE_WER, 2),
    'finetuned_wer':  round(FINAL_WER, 2),
    'abs_improvement': round(abs_improvement, 2),
    'rel_improvement': round(rel_improvement, 1),
}
with open(os.path.join(CFG['output_dir'], 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved to {CFG["output_dir"]}/results.json')

Evaluating fine-tuned model...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step,Wer
0.756624,0.251730,2000,21.310000



  BASELINE WER   : 42.69%
  FINE-TUNED WER : 21.31%
  IMPROVEMENT    : 21.38pp absolute  (50.1% relative)

Results saved to /kaggle/working/whisper-arabic/results.json


In [17]:
SAVE_PATH = os.path.join(CFG['output_dir'], 'final-model')

trainer.save_model(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)

print(f'Model saved to: {SAVE_PATH}')
print('\nFiles:')
for f in sorted(os.listdir(SAVE_PATH)):
    size_mb = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f'  {f:<40s} {size_mb:>8.1f} MB')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /kaggle/working/whisper-arabic/final-model

Files:
  config.json                                   0.0 MB
  generation_config.json                        0.0 MB
  model.safetensors                           967.0 MB
  preprocessor_config.json                      0.0 MB
  processor_config.json                         0.0 MB
  tokenizer.json                                3.9 MB
  tokenizer_config.json                         0.0 MB
  training_args.bin                             0.0 MB


In [18]:
# Show 5 random test samples with ground truth vs prediction

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

print('=== Sample Transcriptions ===')
print(f'{"─"*70}')

# Pick 5 random test samples
indices = np.random.default_rng(0).integers(0, len(df_test_raw), size=5)

for idx in indices:
    row = df_test_raw.iloc[idx]
    try:
        audio, _ = librosa.load(row['audio_path'], sr=16000, mono=True)
        duration  = len(audio) / 16000

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors='pt',
        ).input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(
                inputs,
                language='arabic',
                task='transcribe',
                max_new_tokens=128,
            )

        prediction   = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        ground_truth = row['sentence']

        pred_norm = normalize_arabic(prediction)
        ref_norm  = normalize_arabic(ground_truth)

        # Sample-level WER
        sample_wer = wer_metric.compute(
            predictions=[pred_norm],
            references=[ref_norm],
        ) * 100 if ref_norm else 0.0

        print(f'Duration : {duration:.1f}s')
        print(f'Reference: {ground_truth}')
        print(f'Predicted: {prediction}')
        print(f'WER      : {sample_wer:.1f}%')
        print(f'{"─"*70}')

    except Exception as e:
        print(f'[sample {idx}] Error: {e}')
        print(f'{"─"*70}')

=== Sample Transcriptions ===
──────────────────────────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Duration : 4.3s
Reference: هاملت مسرحيه لشكسبير
Predicted: هاملت مسرحيه لشيكس بير
WER      : 66.7%
──────────────────────────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Duration : 5.3s
Reference: لا تعتقد بانك ستتخلص مني بهذه السهوله
Predicted: لا تعتقد بانك ستتخلص مني بهذه الصهوله
WER      : 14.3%
──────────────────────────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Duration : 3.5s
Reference: نحن نعيش علي كوكب الارض
Predicted: نحن نعيش علي كوكب الارض
WER      : 0.0%
──────────────────────────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Duration : 3.6s
Reference: كانت ليلي تخطط لسرقه مصرف
Predicted: كانت ليلي تخطط لسرقه في مصره
WER      : 40.0%
──────────────────────────────────────────────────────────────────────
Duration : 4.0s
Reference: لم يفعل فاضل شيئا
Predicted: لم يفعل فاضل شيئا
WER      : 0.0%
──────────────────────────────────────────────────────────────────────


In [19]:
print('## Results\n')
print('| Metric               | Value                      |')
print('|----------------------|----------------------------|')
print(f'| Model                | {CFG["model_name"]}  |')
print(f'| Training samples     | {CFG["n_train"]:,}                     |')
print(f'| Test samples         | {CFG["n_test"]:,}                      |')
print(f'| Training steps       | {CFG["max_steps"]:,}                      |')
print(f'| Baseline WER         | {BASELINE_WER:.2f}%                    |')
print(f'| Fine-tuned WER       | {FINAL_WER:.2f}%                    |')
print(f'| Absolute improvement | {BASELINE_WER - FINAL_WER:.2f}pp                   |')
print(f'| Relative improvement | {(BASELINE_WER-FINAL_WER)/BASELINE_WER*100:.1f}%                    |')
print(f'| Dataset              | Common Voice 25.0 Arabic   |')

## Results

| Metric               | Value                      |
|----------------------|----------------------------|
| Model                | openai/whisper-small  |
| Training samples     | 25,000                     |
| Test samples         | 300                      |
| Training steps       | 2,000                      |
| Baseline WER         | 42.69%                    |
| Fine-tuned WER       | 21.31%                    |
| Absolute improvement | 21.38pp                   |
| Relative improvement | 50.1%                    |
| Dataset              | Common Voice 25.0 Arabic   |
